<a id="gh200-cpu"></a>
# 01-1. GH200 — Grace CPU 컴파일과 튜닝

**세션:** 13:30–14:00  
**학습 목표:** 같은 DGEMM 소스 코드를 `GCC + OpenBLAS`와 `nvc + NVPL`의 두 가지 구성으로 빌드하고, 행렬 크기와 스레드 수에 따른 실행 특성을 측정합니다.

이 실습의 비교 단위는 컴파일러와 BLAS 라이브러리를 함께 묶은 **소프트웨어 구성**입니다. 계산 노드에서는 컨테이너에 준비된 도구와 소스만 사용합니다.


## 1. 과정 폴더, Grace CPU, 빌드 환경 확인

`00_Start_Here`의 점검을 통과했다는 전제로, 통합 컨테이너 이미지의 도우미 함수와 CPU 개발 도구를 확인합니다.


In [ ]:
from pathlib import Path
import math
import os
import re
import shutil
import sys

launch_dir = Path.cwd().resolve()
root_candidate = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "gh200" / "notebook_utils.py").is_file()
    ),
    None,
)
if root_candidate is None:
    raise FileNotFoundError("labs/gh200/notebook_utils.py를 찾지 못했습니다.")

REPO_ROOT = root_candidate
LAB_DIR = REPO_ROOT / "labs" / "gh200"
WORK_DIR = REPO_ROOT / "work" / "gh200"
BIN_DIR = WORK_DIR / "bin"
PROFILE_DIR = WORK_DIR / "profiles"
BIN_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from notebook_utils import command_text, print_tool_status, read_image_manifest, run, system_summary

required_tools = ("gcc", "nvc", "make")
if not print_tool_status(required_tools):
    raise EnvironmentError("필수 도구가 없습니다. 계산 노드에서 설치하지 말고 강사에게 알리세요.")

summary = system_summary()
for key, value in summary.items():
    print(f"{key:14s}: {value}")

if str(summary["architecture"]).lower() not in {"aarch64", "arm64"}:
    raise EnvironmentError(f"ARM64 환경이 아닙니다: {summary['architecture']}")
if "GH200" not in str(summary.get("gpu", "")).upper():
    raise EnvironmentError(f"GH200을 확인하지 못했습니다: {summary.get('gpu')}")

manifest = read_image_manifest()
if manifest is None:
    raise FileNotFoundError("통합 image manifest를 찾지 못했습니다: /etc/ksc2026-image.json")
print("이미지 구성 정보: PASS")


## 2. 같은 DGEMM 소스 코드, 서로 다른 빌드 구성

BLAS(Basic Linear Algebra Subprograms)는 행렬·벡터 연산의 표준 인터페이스이며, DGEMM은 배정밀도 행렬 곱셈을 수행합니다. 두 실행 파일은 같은 `dgemm.c`에서 동일한 Fortran BLAS ABI의 `dgemm_` 함수를 호출합니다.

| 실행 파일 | 컴파일러 | BLAS 라이브러리 |
|---|---|---|
| `dgemm-openblas` | GCC | OpenBLAS 0.3.31 |
| `dgemm-nvpl` | NVIDIA HPC SDK `nvc` | NVPL 25.5 |

컴파일러와 BLAS 라이브러리를 동시에 바꾸므로 측정 결과는 두 소프트웨어 구성 전체의 차이를 나타냅니다. 라이브러리만 비교하려면 컴파일러, 빌드 옵션, 스레드 실행 환경, CPU 코어 배치를 동일하게 통제해야 합니다.


In [ ]:
BLAS_DIR = LAB_DIR / "blas"
source_path = BLAS_DIR / "dgemm.c"
makefile_path = BLAS_DIR / "Makefile"
if not source_path.is_file() or not makefile_path.is_file():
    raise FileNotFoundError("DGEMM source 또는 Makefile이 없습니다.")

source_text = source_path.read_text(encoding="utf-8")
for marker in ("extern void dgemm_", "dgemm_(&trans", "checksum"):
    print(f"{'PASS' if marker in source_text else 'FAIL'}  source marker: {marker}")
print(f"Source   : {source_path}")
print(f"Makefile : {makefile_path}")


## 3. 두 구성으로 빌드하고 기준 성능 측정

빌드 결과는 `work/gh200/bin/`에만 저장합니다. 같은 소스 코드와 같은 행렬 입력으로 실행한 뒤, 체크섬이 수치 오차 범위에서 일치하는지 확인합니다.


In [ ]:
make_variables = [
    f"BUILD_DIR={BIN_DIR}",
    "OPENBLAS_PREFIX=/opt/ksc2026/vendor/openblas",
]
run(["make", "clean", *make_variables], cwd=BLAS_DIR)
run(["make", "all", *make_variables], cwd=BLAS_DIR, timeout=600)

OPENBLAS_EXE = BIN_DIR / "dgemm-openblas"
NVPL_EXE = BIN_DIR / "dgemm-nvpl"
for executable in (OPENBLAS_EXE, NVPL_EXE):
    if not executable.is_file():
        raise FileNotFoundError(executable)
print("빌드 결과: PASS")


In [ ]:
def parse_dgemm_output(text):
    pattern = re.compile(
        r"n=(?P<n>\d+) repeats=(?P<repeats>\d+) "
        r"best_seconds=(?P<seconds>[0-9.eE+-]+) "
        r"gflops=(?P<gflops>[0-9.eE+-]+) checksum=(?P<checksum>[0-9.eE+-]+)"
    )
    match = pattern.search(text)
    if match is None:
        raise ValueError(f"DGEMM 결과를 해석할 수 없습니다: {text}")
    values = match.groupdict()
    return {
        "n": int(values["n"]),
        "repeats": int(values["repeats"]),
        "seconds": float(values["seconds"]),
        "gflops": float(values["gflops"]),
        "checksum": float(values["checksum"]),
    }

BASE_N = 2048
BASE_REPEATS = 3
BASE_THREADS = 32
base_env = {
    "OMP_NUM_THREADS": str(BASE_THREADS),
    "OPENBLAS_NUM_THREADS": str(BASE_THREADS),
}

baseline = {}
for stack, executable in (("GCC + OpenBLAS", OPENBLAS_EXE), ("nvc + NVPL", NVPL_EXE)):
    completed = run([executable, str(BASE_N), str(BASE_REPEATS)], env=base_env, timeout=600)
    baseline[stack] = parse_dgemm_output(completed.stdout)

checksums = [result["checksum"] for result in baseline.values()]
CHECKSUM_READY = math.isclose(checksums[0], checksums[1], rel_tol=1e-8, abs_tol=1e-10)
print(f"Checksum comparison: {'PASS' if CHECKSUM_READY else 'FAIL'}")
if not CHECKSUM_READY:
    raise RuntimeError(f"두 stack의 checksum이 일치하지 않습니다: {checksums}")


## 4. 행렬 크기와 CPU 스레드 수 튜닝

`best_seconds`는 같은 조건을 여러 번 실행해 얻은 시간 중 최솟값입니다. 아래 셀은 노드 전체 CPU 수가 아니라 **현재 Slurm 작업에 배정된 CPU 집합** 안에서 스레드 수를 선택합니다.

이 측정은 수업 시간에 행렬 크기와 스레드 수의 영향을 관찰하기 위한 실험입니다. 재현 가능한 공식 벤치마크에는 CPU 코어 배치, 준비 실행(warm-up), 반복 횟수, NUMA 배치와 시스템 부하의 추가 통제가 필요합니다.


In [ ]:
MATRIX_SIZES = [1024, 2048, 4096]

# Linux의 sched_getaffinity는 현재 프로세스가 실제로 실행할 수 있는 CPU만
# 반환합니다. Slurm 값이 함께 있으면 둘 중 더 작은 범위를 사용해
# 노드의 다른 참가자에게 배정된 CPU를 침범하지 않습니다.
affinity_cpus = (
    len(os.sched_getaffinity(0))
    if hasattr(os, "sched_getaffinity")
    else (os.cpu_count() or 1)
)
slurm_cpus_text = os.environ.get("SLURM_CPUS_PER_TASK", "").strip()
if slurm_cpus_text:
    try:
        slurm_cpus = int(slurm_cpus_text)
    except ValueError as exc:
        raise EnvironmentError(
            f"SLURM_CPUS_PER_TASK가 정수가 아닙니다: {slurm_cpus_text!r}"
        ) from exc
    if slurm_cpus < 1:
        raise EnvironmentError(f"SLURM_CPUS_PER_TASK가 올바르지 않습니다: {slurm_cpus}")
    available_cpus = min(affinity_cpus, slurm_cpus)
else:
    available_cpus = affinity_cpus

THREAD_CHOICES = sorted(
    {value for value in (1, 8, 16, 32, available_cpus) if 1 <= value <= available_cpus}
)
TUNING_REPEATS = 2

print(f"Slurm/affinity CPU limit: {available_cpus}")
print(f"Thread sweep             : {THREAD_CHOICES}")

TUNING_RESULTS = []
for n in MATRIX_SIZES:
    for threads in THREAD_CHOICES:
        env = {
            "OMP_NUM_THREADS": str(threads),
            "OPENBLAS_NUM_THREADS": str(threads),
        }
        for stack, executable in (("OpenBLAS", OPENBLAS_EXE), ("NVPL", NVPL_EXE)):
            completed = run([executable, str(n), str(TUNING_REPEATS)], env=env, timeout=600)
            result = parse_dgemm_output(completed.stdout)
            TUNING_RESULTS.append({"stack": stack, "threads": threads, **result})

print(f"\n{'Stack':10s} {'N':>6s} {'Threads':>8s} {'Seconds':>10s} {'GFLOPS':>10s}")
for result in TUNING_RESULTS:
    print(
        f"{result['stack']:10s} {result['n']:6d} {result['threads']:8d} "
        f"{result['seconds']:10.4f} {result['gflops']:10.1f}"
    )

print("\n각 stack의 관찰상 최고 GFLOPS")
for stack in ("OpenBLAS", "NVPL"):
    best = max(
        (row for row in TUNING_RESULTS if row["stack"] == stack),
        key=lambda row: row["gflops"],
    )
    print(
        f"{stack:10s}: N={best['n']}, threads={best['threads']}, "
        f"{best['gflops']:.1f} GFLOPS"
    )


### 튜닝 결과를 살펴볼 질문

1. 작은 행렬과 큰 행렬에서 스레드 수를 늘렸을 때의 효과가 같았나요?
2. 어느 지점부터 스레드를 더 늘려도 성능이 좋아지지 않았나요?
3. 두 소프트웨어 구성에서 가장 높은 성능이 나온 조건이 같았나요?
4. 결과를 재현할 수 있는 벤치마크로 만들려면 CPU 코어 배치, 준비 실행(warm-up), 반복 횟수, 시스템 부하를 어떻게 통제해야 할까요?


## 5. 결과 정리와 완료 확인

- 체크섬 일치는 두 소프트웨어 구성이 수치 오차 범위에서 같은 계산 결과를 냈음을 뜻합니다.
- GFLOP/s는 DGEMM의 부동소수점 연산 처리율이며, 행렬 크기와 스레드 수에 따라 달라집니다.
- 이번 결과는 현재 Slurm 할당과 실행 조건에서 얻은 수업용 측정값입니다.

**완료 체크리스트**

- [ ] `dgemm-openblas`와 `dgemm-nvpl` 실행 파일을 만들었습니다.
- [ ] 두 실행 파일의 체크섬 일치를 확인했습니다.
- [ ] 각 소프트웨어 구성에서 가장 높은 GFLOP/s가 나온 행렬 크기와 스레드 수를 기록했습니다.
- [ ] 결과를 재현하려면 추가로 통제해야 할 조건을 설명할 수 있습니다.

다음으로 [02_GPU_Memory_Profile.ipynb](02_GPU_Memory_Profile.ipynb)에서 Hopper GPU의 메모리 경로와 실행 순서를 확인합니다.


---

## 출처와 라이선스

이 실습은 KSC 2026을 위해 별도로 작성한 소스 코드를 사용합니다. OpenBLAS, NVIDIA 개발 도구, NVPL에는 각 원저작물의 라이선스와 고지가 적용됩니다.
